# Hierarchical Bayes and Partial Pooling

## Historical problem

The modelling revolution in Bayesian statistics expanded uncertainty from isolated parameters to structured families of related parameters. Hierarchical Bayes is the clearest example: group-level parameters are allowed to vary, but they are tied together by shared higher-level uncertainty.

This notebook uses the classic eight-schools example to compare no pooling, complete pooling, and hierarchical partial pooling.

In [ ]:
from pathlib import Path
import sys

import arviz as az
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()

## Eight schools data

The data record estimated coaching effects and known standard errors for eight schools:

- observed treatment effects `y_j`,
- known standard errors `sigma_j`.

No pooling treats each school separately, complete pooling treats them as identical, and the hierarchical model allows partial pooling.

In [ ]:
school_names = ["A", "B", "C", "D", "E", "F", "G", "H"]
y = np.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])
sigma = np.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])

no_pooling = y
complete_pooling = np.average(y, weights=1.0 / sigma**2)

with pm.Model() as hierarchical_model:
    mu = pm.Normal("mu", mu=0.0, sigma=10.0)
    tau = pm.HalfNormal("tau", sigma=10.0)
    eta = pm.Normal("eta", mu=0.0, sigma=1.0, shape=len(y))
    theta = pm.Deterministic("theta", mu + tau * eta)
    pm.Normal("obs", mu=theta, sigma=sigma, observed=y)
    trace = pm.sample(1000, tune=1500, chains=2, cores=1, random_seed=42, progressbar=False, target_accept=0.95)

theta_samples = trace.posterior["theta"].values.reshape(-1, len(y))
theta_post = theta_samples.mean(axis=0)
intervals = np.quantile(theta_samples, [0.025, 0.975], axis=0).T

print("Complete-pooling estimate:", round(float(complete_pooling), 2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
y_pos = np.arange(len(y))

ax.errorbar(no_pooling, y_pos + 0.15, xerr=0.0, fmt="o", color="#111111", label="No pooling")
ax.errorbar(np.repeat(complete_pooling, len(y)), y_pos, xerr=0.0, fmt="o", color="#54a24b", label="Complete pooling")
ax.errorbar(
    theta_post,
    y_pos - 0.15,
    xerr=[theta_post - intervals[:, 0], intervals[:, 1] - theta_post],
    fmt="o",
    color="#d62728",
    ecolor="#d62728",
    capsize=3,
    label="Hierarchical posterior mean and 95% HDI",
)

ax.set_yticks(y_pos)
ax.set_yticklabels(school_names)
ax.set_xlabel("Estimated effect")
ax.set_title("Partial pooling shrinks school estimates toward a common level")
ax.legend(loc="lower right")
fig.tight_layout()
save_fig(fig, Path("figs") / "hierarchical_partial_pooling.png")
plt.show()

## Interpretation

The hierarchical estimates lie between the extremes:

- they are less noisy than no-pooling estimates,
- but less rigid than complete pooling.

That shrinkage effect is the signature of partial pooling and one of the clearest practical reasons Bayesian hierarchical models became so important.

## References

- Gelman et al. (2013), *Bayesian Data Analysis*.
- Standard literature on hierarchical modelling and partial pooling.